<a href="https://colab.research.google.com/github/RafaelMuniz94/RaruRevival/blob/main/Museu_do_Raru_Cutter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Museu do Raru — Cutter

Notebook para a **Fase 2** do Museu do Raru.

Ele usa a estrutura já existente do projeto:

```text
RaruRevival/
├── Lives/
├── Transcriptions/
├── Models/
├── Logs/
├── Temp/
├── Checkpoints/
├── Manifests/
└── Cuts/
```

## Como funciona

1. Monta o Google Drive.
2. Cria automaticamente `Manifests/` e `Cuts/` caso não existam.
3. Procura primeiro os vídeos em `RaruRevival/Lives/`.
4. Se não houver nenhum vídeo, encerra sem procurar manifests.
5. Para cada vídeo, procura um JSON com o mesmo nome-base em `RaruRevival/Manifests/`.

Exemplo:

```text
Lives/Amanda 1.mp4
Manifests/Amanda 1.json
```

6. Valida o manifesto e os timestamps.
7. Gera um MP4 por momento em:

```text
Cuts/Amanda 1/
```

8. Preserva resolução e FPS e usa reencode H.264 de alta qualidade para permitir cortes precisos.
9. Gera um relatório JSON e, opcionalmente, um ZIP por live.

> Fluxo recomendado no Colab: **Ambiente de execução → Executar tudo**.


In [ ]:
# ============================================================
# CÉLULA 1 — DRIVE, ESTRUTURA E CONFIGURAÇÃO
# ============================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime
import os
import shutil

drive.mount("/content/drive")

# ------------------------------------------------------------
# RAIZ DO PROJETO
# ------------------------------------------------------------

RARU_REVIVAL_ROOT = Path("/content/drive/MyDrive/RaruRevival")

# Pastas já existentes
LIVES_DIR = RARU_REVIVAL_ROOT / "Lives"
TRANSCRIPTIONS_DIR = RARU_REVIVAL_ROOT / "Transcriptions"
MODELS_DIR = RARU_REVIVAL_ROOT / "Models"
LOGS_DIR = RARU_REVIVAL_ROOT / "Logs"
TEMP_DIR = RARU_REVIVAL_ROOT / "Temp"
CHECKPOINTS_DIR = RARU_REVIVAL_ROOT / "Checkpoints"

# Pastas do Museu
MANIFESTS_DIR = RARU_REVIVAL_ROOT / "Manifests"
CUTS_DIR = RARU_REVIVAL_ROOT / "Cuts"

# Cria a estrutura, inclusive as novas pastas
PROJECT_DIRS = [
    LIVES_DIR,
    TRANSCRIPTIONS_DIR,
    MODELS_DIR,
    LOGS_DIR,
    TEMP_DIR,
    CHECKPOINTS_DIR,
    MANIFESTS_DIR,
    CUTS_DIR,
]

for directory in PROJECT_DIRS:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# CONFIGURAÇÕES
# ------------------------------------------------------------

# Qualidade: menor CRF = maior qualidade e arquivo maior.
# 14 é uma configuração de altíssima qualidade.
VIDEO_CRF = 14

# ultrafast, veryfast, fast, medium, slow, slower, veryslow
PRESET = "slow"

AUDIO_BITRATE = "320k"

# True = substitui cortes existentes.
OVERWRITE = True

# True = cria um ZIP de cada live processada.
CREATE_ZIP = True

# Se False, arquivo já existente será pulado quando OVERWRITE=False.
# Se True, um erro em um momento não impede os demais.
CONTINUE_ON_ERROR = True

VIDEO_EXTENSIONS = {
    ".mp4", ".mkv", ".mov", ".m4v", ".webm", ".avi"
}

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_LOG = LOGS_DIR / f"MuseuCutter_{RUN_ID}.log"

print("=" * 72)
print("RARU REVIVAL / MUSEU DO RARU — CUTTER")
print("=" * 72)
print(f"Raiz:       {RARU_REVIVAL_ROOT}")
print(f"Lives:      {LIVES_DIR}")
print(f"Manifests:  {MANIFESTS_DIR}")
print(f"Cortes:     {CUTS_DIR}")
print(f"Log:        {RUN_LOG}")
print()
print("Estrutura pronta.")


Mounted at /content/drive
RARU REVIVAL / MUSEU DO RARU — CUTTER
Raiz:       /content/drive/MyDrive/RaruRevival
Lives:      /content/drive/MyDrive/RaruRevival/Lives
Manifests:  /content/drive/MyDrive/RaruRevival/Manifests
Cortes:     /content/drive/MyDrive/RaruRevival/Cuts
Log:        /content/drive/MyDrive/RaruRevival/Logs/MuseuCutter_20261006_003558.log

Estrutura pronta.


In [ ]:
# ============================================================
# CÉLULA 2 — MOTOR DO CUTTER
# ============================================================

import json
import re
import subprocess
import unicodedata
from fractions import Fraction


def log(message=""):
    """Imprime e registra no arquivo de log da execução."""
    text = str(message)
    print(text)
    with open(RUN_LOG, "a", encoding="utf-8") as f:
        f.write(text + "\n")


def ensure_ffmpeg():
    """Garante que ffmpeg e ffprobe estejam disponíveis."""
    if shutil.which("ffmpeg") and shutil.which("ffprobe"):
        log("FFmpeg/ffprobe encontrados.")
        return

    log("FFmpeg não encontrado. Instalando...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
    log("FFmpeg instalado.")


def timestamp_to_seconds(timestamp):
    """Converte HH:MM:SS.mmm em segundos."""
    if not isinstance(timestamp, str):
        raise ValueError(f"Timestamp inválido: {timestamp!r}")

    match = re.fullmatch(r"(\d+):([0-5]\d):([0-5]\d(?:\.\d{1,6})?)", timestamp.strip())
    if not match:
        raise ValueError(f"Timestamp inválido: {timestamp}")

    hours = int(match.group(1))
    minutes = int(match.group(2))
    seconds = float(match.group(3))
    return hours * 3600 + minutes * 60 + seconds


def seconds_to_timestamp(seconds):
    """Converte segundos em HH:MM:SS.mmm."""
    seconds = max(0.0, float(seconds))
    hours = int(seconds // 3600)
    seconds -= hours * 3600
    minutes = int(seconds // 60)
    seconds -= minutes * 60
    return f"{hours:02d}:{minutes:02d}:{seconds:06.3f}"


def sanitize_filename(text, max_length=90):
    """Cria slug seguro para nome de arquivo."""
    text = unicodedata.normalize("NFKD", str(text))
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return (text or "momento")[:max_length].rstrip("-")


def timestamp_for_filename(timestamp):
    """02:18:32.400 -> 02-18-32"""
    seconds = timestamp_to_seconds(timestamp)
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}-{minutes:02d}-{secs:02d}"


def ffprobe(path):
    cmd = [
        "ffprobe",
        "-v", "error",
        "-show_format",
        "-show_streams",
        "-of", "json",
        str(path),
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        check=True,
    )

    return json.loads(result.stdout)


def get_video_info(path):
    data = ffprobe(path)

    video_stream = next(
        (s for s in data.get("streams", []) if s.get("codec_type") == "video"),
        None,
    )
    audio_stream = next(
        (s for s in data.get("streams", []) if s.get("codec_type") == "audio"),
        None,
    )

    if video_stream is None:
        raise RuntimeError(f"Nenhum stream de vídeo encontrado em: {path}")

    duration = float(data.get("format", {}).get("duration", 0) or 0)
    width = video_stream.get("width")
    height = video_stream.get("height")

    fps_raw = video_stream.get("avg_frame_rate") or video_stream.get("r_frame_rate")
    try:
        fps = float(Fraction(fps_raw)) if fps_raw and fps_raw != "0/0" else None
    except Exception:
        fps = None

    return {
        "duration": duration,
        "width": width,
        "height": height,
        "resolution": f"{width}x{height}",
        "fps": fps,
        "video_codec": video_stream.get("codec_name"),
        "pixel_format": video_stream.get("pix_fmt"),
        "audio_codec": audio_stream.get("codec_name") if audio_stream else None,
        "has_audio": audio_stream is not None,
    }


def load_manifest(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        manifest = json.load(f)

    if not isinstance(manifest, dict):
        raise ValueError("O manifest deve conter um objeto JSON na raiz.")

    moments = manifest.get("moments")

    if not isinstance(moments, list):
        raise ValueError("O manifest não contém uma lista 'moments'.")

    if not moments:
        raise ValueError("O manifest possui 'moments', mas a lista está vazia.")

    return manifest


def validate_manifest(manifest, source_info):
    """Retorna (erros, avisos)."""
    errors = []
    warnings = []
    video_duration = source_info["duration"]

    required = ["id", "rank", "title", "start", "end"]
    ranks_seen = set()

    for index, moment in enumerate(manifest["moments"], start=1):
        label = moment.get("id", f"momento #{index}")

        for field in required:
            if field not in moment:
                errors.append(f"{label}: campo obrigatório ausente: '{field}'.")

        if "rank" in moment:
            try:
                rank = int(moment["rank"])
                if rank in ranks_seen:
                    errors.append(f"{label}: rank duplicado: {rank}.")
                ranks_seen.add(rank)
            except Exception:
                errors.append(f"{label}: rank inválido: {moment['rank']!r}.")

        if "start" not in moment or "end" not in moment:
            continue

        try:
            start = timestamp_to_seconds(moment["start"])
            end = timestamp_to_seconds(moment["end"])
        except Exception as exc:
            errors.append(f"{label}: {exc}")
            continue

        if end <= start:
            errors.append(f"{label}: END precisa ser maior que START.")

        if start >= video_duration:
            errors.append(
                f"{label}: START {moment['start']} está fora do vídeo "
                f"({seconds_to_timestamp(video_duration)})."
            )

        if end > video_duration + 0.250:
            errors.append(
                f"{label}: END {moment['end']} está fora do vídeo "
                f"({seconds_to_timestamp(video_duration)})."
            )

        calculated = end - start

        if "duration_seconds" in moment:
            try:
                declared = float(moment["duration_seconds"])
                difference = abs(calculated - declared)

                if difference > 0.100:
                    warnings.append(
                        f"{label}: duration_seconds={declared:.3f}s, "
                        f"mas START/END resultam em {calculated:.3f}s."
                    )
            except Exception:
                warnings.append(
                    f"{label}: duration_seconds não pôde ser interpretado."
                )

    return errors, warnings


def make_output_filename(moment):
    rank = int(moment["rank"])
    slug = sanitize_filename(moment["title"])
    ts = timestamp_for_filename(moment["start"])
    return f"MUSEU_{rank:02d}_{ts}_{slug}.mp4"


def cut_moment(video_path, moment, output_dir, source_info):
    start = timestamp_to_seconds(moment["start"])
    end = timestamp_to_seconds(moment["end"])
    duration = end - start

    filename = make_output_filename(moment)
    output_path = output_dir / filename

    if output_path.exists() and not OVERWRITE:
        log(f"  ↳ Pulado: {filename} já existe.")
        return {
            "id": moment.get("id"),
            "rank": int(moment["rank"]),
            "title": moment["title"],
            "file": filename,
            "start": moment["start"],
            "end": moment["end"],
            "status": "PULADO_EXISTENTE",
        }

    overwrite_flag = "-y" if OVERWRITE else "-n"

    # Seek antes da entrada + reencode = corte preciso, sem depender de keyframe.
    # Não há scale/crop/filtro: resolução e proporção permanecem as da fonte.
    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        overwrite_flag,
        "-ss", f"{start:.6f}",
        "-i", str(video_path),
        "-t", f"{duration:.6f}",
        "-map", "0:v:0",
        "-map", "0:a:0?",
        "-map_metadata", "0",
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(VIDEO_CRF),
        "-fps_mode", "passthrough",
        "-c:a", "aac",
        "-b:a", AUDIO_BITRATE,
        "-movflags", "+faststart",
        "-avoid_negative_ts", "make_zero",
        str(output_path),
    ]

    subprocess.run(cmd, check=True)

    output_info = get_video_info(output_path)
    actual_duration = output_info["duration"]
    duration_error = abs(actual_duration - duration)

    same_resolution = (
        output_info["width"] == source_info["width"]
        and output_info["height"] == source_info["height"]
    )

    fps_ok = True
    if source_info["fps"] and output_info["fps"]:
        fps_ok = abs(source_info["fps"] - output_info["fps"]) < 0.10

    # Alguns contêineres/áudios introduzem poucos ms de diferença.
    duration_ok = duration_error <= 0.200

    status = "OK" if same_resolution and fps_ok and duration_ok else "VERIFICAR"

    return {
        "id": moment.get("id"),
        "rank": int(moment["rank"]),
        "title": moment["title"],
        "score": moment.get("score"),
        "category": moment.get("category", []),
        "file": filename,
        "start": moment["start"],
        "end": moment["end"],
        "target_duration_seconds": round(duration, 3),
        "actual_duration_seconds": round(actual_duration, 3),
        "duration_error_ms": round(duration_error * 1000, 2),
        "resolution": output_info["resolution"],
        "fps": round(output_info["fps"], 3) if output_info["fps"] else None,
        "video_codec": output_info["video_codec"],
        "status": status,
    }


def process_job(job):
    video_path = job["video_path"]
    manifest_path = job["manifest_path"]
    output_dir = job["output_dir"]
    output_dir.mkdir(parents=True, exist_ok=True)

    log("")
    log("=" * 72)
    log(f"PROCESSANDO: {job['source_name']}")
    log("=" * 72)
    log(f"Vídeo:    {video_path}")
    log(f"Manifest: {manifest_path}")
    log(f"Saída:    {output_dir}")

    manifest = load_manifest(manifest_path)
    source_info = get_video_info(video_path)

    log(
        f"Fonte: {source_info['resolution']} | "
        f"{source_info['fps']:.3f} FPS | "
        f"{seconds_to_timestamp(source_info['duration'])}"
        if source_info["fps"]
        else
        f"Fonte: {source_info['resolution']} | FPS desconhecido | "
        f"{seconds_to_timestamp(source_info['duration'])}"
    )

    # Proteção extra: o campo source não decide a associação, mas pode alertar.
    manifest_source = str(manifest.get("source", "")).strip()
    if manifest_source and manifest_source.casefold() != job["source_name"].casefold():
        log(
            f"AVISO: manifest.source='{manifest_source}' difere do nome-base "
            f"do vídeo '{job['source_name']}'. A associação pelo nome de arquivo foi mantida."
        )

    errors, warnings = validate_manifest(manifest, source_info)

    for warning in warnings:
        log(f"AVISO: {warning}")

    if errors:
        for error in errors:
            log(f"ERRO: {error}")
        raise RuntimeError(
            f"Manifest inválido para '{job['source_name']}'. "
            f"{len(errors)} erro(s) encontrado(s)."
        )

    moments = sorted(manifest["moments"], key=lambda x: int(x["rank"]))
    cuts_report = []

    for moment in moments:
        rank = int(moment["rank"])
        log("")
        log(
            f"[{rank:02d}] {moment['title']} | "
            f"{moment['start']} → {moment['end']}"
        )

        try:
            result = cut_moment(
                video_path=video_path,
                moment=moment,
                output_dir=output_dir,
                source_info=source_info,
            )
            cuts_report.append(result)

            if result["status"] == "PULADO_EXISTENTE":
                continue

            log(
                f"  ↳ {result['file']} | "
                f"{result['actual_duration_seconds']:.3f}s | "
                f"{result['resolution']} | "
                f"{result['fps']} FPS | "
                f"{result['status']}"
            )

        except Exception as exc:
            error_result = {
                "id": moment.get("id"),
                "rank": rank,
                "title": moment.get("title"),
                "start": moment.get("start"),
                "end": moment.get("end"),
                "status": "ERRO",
                "error": str(exc),
            }
            cuts_report.append(error_result)
            log(f"  ↳ ERRO: {exc}")

            if not CONTINUE_ON_ERROR:
                raise

    final_report = {
        "project": manifest.get("project", "Museu do Raru"),
        "source": manifest.get("source", job["source_name"]),
        "video": str(video_path),
        "manifest": str(manifest_path),
        "source_info": source_info,
        "settings": {
            "video_encoder": "libx264",
            "crf": VIDEO_CRF,
            "preset": PRESET,
            "audio_codec": "aac",
            "audio_bitrate": AUDIO_BITRATE,
        },
        "cuts": cuts_report,
    }

    report_path = output_dir / "MUSEU_CUT_REPORT.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(final_report, f, indent=2, ensure_ascii=False)

    zip_path = None
    if CREATE_ZIP:
        zip_base = output_dir.parent / f"{output_dir.name}_MUSEU"
        zip_path = Path(
            shutil.make_archive(
                str(zip_base),
                "zip",
                root_dir=output_dir,
            )
        )

    ok_count = sum(1 for x in cuts_report if x.get("status") == "OK")
    verify_count = sum(1 for x in cuts_report if x.get("status") == "VERIFICAR")
    skipped_count = sum(1 for x in cuts_report if x.get("status") == "PULADO_EXISTENTE")
    error_count = sum(1 for x in cuts_report if x.get("status") == "ERRO")

    log("")
    log(
        f"Resultado: {ok_count} OK | {verify_count} verificar | "
        f"{skipped_count} pulados | {error_count} erros"
    )
    log(f"Relatório: {report_path}")
    if zip_path:
        log(f"ZIP:       {zip_path}")

    return {
        "source_name": job["source_name"],
        "report_path": str(report_path),
        "zip_path": str(zip_path) if zip_path else None,
        "cuts": cuts_report,
    }


ensure_ffmpeg()
log("Motor do Museu Cutter carregado.")


FFmpeg/ffprobe encontrados.
Motor do Museu Cutter carregado.


In [ ]:
# ============================================================
# CÉLULA 3 — DESCOBRIR LIVES E ASSOCIAR MANIFESTS
# ============================================================
#
# REGRA IMPORTANTE:
# 1. Primeiro procura vídeos.
# 2. Se não encontrar nenhum vídeo, NÃO procura manifests.
# 3. Somente depois associa cada vídeo ao JSON de mesmo nome-base.
# ============================================================

video_files = sorted(
    [
        file
        for file in LIVES_DIR.iterdir()
        if file.is_file() and file.suffix.lower() in VIDEO_EXTENSIONS
    ],
    key=lambda p: p.name.casefold(),
)

HAS_VIDEOS = bool(video_files)
jobs = []
videos_without_manifest = []

log("")
log("=" * 72)
log("DESCOBERTA DE LIVES")
log("=" * 72)

if not HAS_VIDEOS:
    log(f"Nenhum vídeo encontrado em: {LIVES_DIR}")
    log("A busca por manifests NÃO será executada.")
    log("Execução encerrada antes da etapa de manifests.")

else:
    log(f"{len(video_files)} live(s) encontrada(s):")
    for i, video in enumerate(video_files, start=1):
        log(f"  {i:02d}. {video.name}")

    # Só entra nesta etapa porque pelo menos um vídeo existe.
    log("")
    log("=" * 72)
    log("ASSOCIAÇÃO LIVE ↔ MANIFEST")
    log("=" * 72)

    for video_path in video_files:
        source_name = video_path.stem

        # Correspondência EXATA pelo nome-base:
        # Lives/Amanda 1.mp4 -> Manifests/Amanda 1.json
        manifest_path = MANIFESTS_DIR / f"{source_name}.json"

        if not manifest_path.is_file():
            videos_without_manifest.append(video_path)
            log(f"SEM MANIFEST: {video_path.name} → esperado: {manifest_path.name}")
            continue

        output_dir = CUTS_DIR / source_name

        jobs.append(
            {
                "source_name": source_name,
                "video_path": video_path,
                "manifest_path": manifest_path,
                "output_dir": output_dir,
            }
        )

        log(f"PRONTO: {video_path.name} ↔ {manifest_path.name}")

    log("")
    log(f"Jobs prontos: {len(jobs)}")
    log(f"Lives sem manifest: {len(videos_without_manifest)}")

    if not jobs:
        log("Nenhum par vídeo/manifest encontrado. Nenhum corte será realizado.")



DESCOBERTA DE LIVES
1 live(s) encontrada(s):
  01. Amanda 1.mp4

ASSOCIAÇÃO LIVE ↔ MANIFEST
PRONTO: Amanda 1.mp4 ↔ Amanda 1.json

Jobs prontos: 1
Lives sem manifest: 0


In [ ]:
# ============================================================
# CÉLULA 4 — PROCESSAR TODOS OS JOBS PRONTOS
# ============================================================

all_results = []

if not HAS_VIDEOS:
    print("Nada a processar: nenhuma live foi encontrada.")

elif not jobs:
    print("Nada a processar: nenhuma live possui manifest correspondente.")

else:
    print("=" * 72)
    print(f"INICIANDO {len(jobs)} JOB(S)")
    print("=" * 72)

    for job in jobs:
        try:
            result = process_job(job)
            all_results.append(result)
        except Exception as exc:
            log("")
            log(f"FALHA NO JOB '{job['source_name']}': {exc}")

            all_results.append(
                {
                    "source_name": job["source_name"],
                    "status": "ERRO_JOB",
                    "error": str(exc),
                }
            )

            if not CONTINUE_ON_ERROR:
                raise

    print()
    print("=" * 72)
    print("RESUMO FINAL")
    print("=" * 72)

    for result in all_results:
        if result.get("status") == "ERRO_JOB":
            print(f"❌ {result['source_name']}: erro no job")
            continue

        cuts = result.get("cuts", [])
        ok = sum(1 for x in cuts if x.get("status") == "OK")
        verify = sum(1 for x in cuts if x.get("status") == "VERIFICAR")
        skipped = sum(1 for x in cuts if x.get("status") == "PULADO_EXISTENTE")
        errors = sum(1 for x in cuts if x.get("status") == "ERRO")

        print(
            f"✅ {result['source_name']}: "
            f"{ok} OK | {verify} verificar | {skipped} pulados | {errors} erros"
        )

    print()
    print(f"Log completo: {RUN_LOG}")


INICIANDO 1 JOB(S)

PROCESSANDO: Amanda 1
Vídeo:    /content/drive/MyDrive/RaruRevival/Lives/Amanda 1.mp4
Manifest: /content/drive/MyDrive/RaruRevival/Manifests/Amanda 1.json
Saída:    /content/drive/MyDrive/RaruRevival/Cuts/Amanda 1
Fonte: 1280x720 | 30.000 FPS | 03:16:15.884

[01] A resposta errada vira uma morte bizarra | 02:18:32.400 → 02:19:07.180
  ↳ MUSEU_01_02-18-32_a-resposta-errada-vira-uma-morte-bizarra.mp4 | 34.823s | 1280x720 | 30.0 FPS | OK

[02] O porco surge do nada | 02:12:54.920 → 02:13:09.520
  ↳ MUSEU_02_02-12-54_o-porco-surge-do-nada.mp4 | 14.623s | 1280x720 | 30.0 FPS | OK

[03] Eu ouvi um barulho... | 01:41:14.500 → 01:41:38.020
  ↳ MUSEU_03_01-41-14_eu-ouvi-um-barulho.mp4 | 23.556s | 1280x720 | 30.0 FPS | OK

[04] O braço aparece e o Raru congela | 00:43:20.560 → 00:44:11.460
  ↳ MUSEU_04_00-43-20_o-braco-aparece-e-o-raru-congela.mp4 | 50.923s | 1280x720 | 30.0 FPS | OK

[05] O jogo vira sozinho e ele acha que morreu | 00:35:01.000 → 00:35:56.600
  ↳ MUSEU_05_00

## Onde colocar os manifests

Use:

```text
/content/drive/MyDrive/RaruRevival/Manifests/
```

O nome-base deve ser igual ao vídeo.

Exemplos válidos:

```text
Lives/Amanda 1.mp4
Manifests/Amanda 1.json
```

```text
Lives/Silent Hill - Parte 03.mkv
Manifests/Silent Hill - Parte 03.json
```

A extensão do vídeo pode variar; o nome-base é o que determina a associação.

## Saída

Para `Amanda 1.mp4`:

```text
RaruRevival/Cuts/Amanda 1/
├── MUSEU_01_02-18-32_a-resposta-errada-vira-uma-morte-bizarra.mp4
├── MUSEU_02_02-12-54_o-porco-surge-do-nada.mp4
├── ...
└── MUSEU_CUT_REPORT.json
```

Também é criado, se `CREATE_ZIP = True`:

```text
RaruRevival/Cuts/Amanda 1_MUSEU.zip
```

Os logs da execução ficam em:

```text
RaruRevival/Logs/MuseuCutter_YYYYMMDD_HHMMSS.log
```
